## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Oil, fuel and tech share prices: cointegration test

**Cointegration Test** · a worked example from the StatNex teaching guide

**The story.** Fifty weeks of prices: crude oil, gasoline and jet fuel (both made from crude oil), plus a tech company's share price. Each series wanders on its own.

**The question.** Is crude oil tied to the other three prices in the long run, so that the gap between them stays stable?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/cointegration_test.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Week,Crude oil ($/barrel),Gasoline ($/gallon),Jet fuel ($/gallon),Tech stock ($)
1.0,72.63,2.717,2.7,122.95
2.0,71.22,2.667,2.655,120.71
3.0,69.18,2.657,2.567,123.61
4.0,71.71,2.694,2.637,122.51
5.0,72.18,2.705,2.648,120.36
6.0,72.76,2.709,2.674,122.2
7.0,70.18,2.701,2.605,125.85
8.0,68.3,2.574,2.542,131.26
9.0,64.71,2.462,2.395,132.51
10.0,63.82,2.505,2.365,138.73
11.0,63.07,2.443,2.379,137.56
12.0,63.61,2.483,2.423,138.2
13.0,65.24,2.524,2.421,139.03
14.0,63.44,2.457,2.372,139.84
15.0,61.27,2.407,2.264,138.32
16.0,59.9,2.352,2.264,138.44
17.0,57.59,2.292,2.146,139.48
18.0,57.22,2.237,2.181,141.24
19.0,53.69,2.171,2.076,139.26
20.0,55.09,2.19,2.098,133.38
21.0,54.38,2.155,2.058,131.61
22.0,56.98,2.264,2.17,133.63
23.0,54.99,2.22,2.122,136.08
24.0,55.12,2.184,2.14,135.92
25.0,54.74,2.207,2.134,134.98
26.0,53.55,2.156,2.061,133.26
27.0,54.03,2.18,2.093,134.01
28.0,54.02,2.174,2.075,130.79
29.0,52.82,2.146,2.029,131.37
30.0,54.08,2.157,2.059,130.24
31.0,56.73,2.255,2.152,132.27
32.0,55.75,2.238,2.144,136.51
33.0,55.22,2.217,2.167,132.7
34.0,51.87,2.127,2.008,128.8
35.0,52.08,2.143,2.029,127.73
36.0,52.03,2.138,1.985,124.06
37.0,53.07,2.107,2.05,127.25
38.0,53.63,2.152,2.049,126.7
39.0,50.9,2.075,1.97,124.41
40.0,54.81,2.23,2.127,118.66
41.0,57.35,2.243,2.218,115.02
42.0,60.64,2.348,2.292,117.93
43.0,61.02,2.339,2.296,121.0
44.0,64.37,2.513,2.42,116.71
45.0,65.82,2.523,2.411,115.82
46.0,68.68,2.594,2.517,111.02
47.0,67.02,2.573,2.501,112.39
48.0,67.56,2.581,2.518,110.69
49.0,68.36,2.615,2.509,107.66
50.0,65.85,2.531,2.49,106.28
""")

## Helper: cointegration_test

A step that no library does, written out. In the code download it is the file `cointegration_test.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The P value of the Phillips-Ouliaris Z(alpha) statistic, read off a simulated table.

Under "no cointegration" the statistic's distribution does not depend on the data, only
on how many regressors the cointegrating regression has and whether it has an
intercept, so it can be tabulated once. These quantiles are StatNex's own: 60,000
simulated systems of length 2,000 for each regressor count (the 1-regressor 5% point
with an intercept, -20.44, agrees with Phillips & Ouliaris 1990, Table Ia, -20.5).
No Python library ships them.
"""

import numpy as np

PROBABILITIES = [0.001, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
                 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]

#: Regression through the origin, by number of regressors.
WITHOUT_INTERCEPT = {
    1: [-32.3226, -25.7551, -22.8289, -18.7736, -15.8205, -12.6497, -8.2789,
        -4.6585, -2.1379, -0.5015, 0.2945, 0.8836, 1.4989, 1.9492, 2.8180],
    2: [-41.9730, -33.5725, -30.1018, -25.6898, -22.1247, -18.5237, -13.2796,
        -8.7110, -5.1886, -2.7838, -1.6299, -0.7285, 0.3243, 0.9591, 2.1984],
    3: [-48.7671, -40.6098, -36.8053, -32.0054, -28.2021, -24.1368, -18.1820,
        -12.6731, -8.3834, -5.3370, -3.8251, -2.6942, -1.4480, -0.5356, 1.0067],
    4: [-55.3672, -46.6590, -43.0281, -37.9877, -33.8722, -29.4313, -22.7836,
        -16.6881, -11.7297, -8.0358, -6.2434, -4.8105, -3.3343, -2.4812, -0.5085],
    5: [-61.3899, -53.0552, -49.2189, -43.6191, -39.1145, -34.4641, -27.3718,
        -20.6555, -15.0324, -10.8444, -8.7899, -7.1544, -5.5637, -4.5232, -2.5539],
}

#: Regression with an intercept (the usual "demeaned" variant).
WITH_INTERCEPT = {
    1: [-37.7894, -30.8949, -27.8737, -23.7955, -20.4383, -16.9899, -12.1489,
        -7.9325, -4.7501, -2.6018, -1.4993, -0.6694, 0.2185, 0.7800, 1.8920],
    2: [-46.4847, -38.4390, -34.9964, -30.2986, -26.5207, -22.5893, -16.8740,
        -11.7931, -7.7452, -4.9306, -3.4878, -2.3460, -1.1692, -0.4212, 1.0270],
    3: [-53.7778, -44.8609, -41.2403, -36.0915, -32.1141, -27.8818, -21.5588,
        -15.6954, -10.9896, -7.5921, -5.8442, -4.4695, -3.0062, -2.0177, -0.2216],
    4: [-60.5073, -51.8089, -47.6683, -42.3455, -37.8854, -33.1177, -26.1629,
        -19.6798, -14.3227, -10.3572, -8.2958, -6.7039, -4.9937, -3.9152, -2.0167],
    5: [-66.7558, -56.9690, -52.8583, -47.2598, -42.9225, -38.0759, -30.6436,
        -23.5325, -17.6234, -13.2335, -10.8349, -9.0892, -7.0130, -5.6861, -3.4439],
}


def phillips_ouliaris_p(statistic, regressors, intercept):
    """(P, note): linear interpolation in the table for this many regressors (at most 5).

    The P value is only reported between 1% and 99%; beyond, the note says so.
    """
    table = WITH_INTERCEPT if intercept else WITHOUT_INTERCEPT
    quantiles = table[min(regressors, 5)]
    p = float(np.interp(statistic, quantiles, PROBABILITIES))
    note = ("below 0.01" if statistic < quantiles[2]
            else "above 0.99" if statistic > quantiles[-3] else "")
    return p, note

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
# phillips_ouliaris_p: defined in the helper cell cointegration_test above (cointegration_test.py in the code download)

pd.set_option("display.width", 120)

## The data

In [ ]:
# Two to six time series of the same length, in row order, with no blank cells. The
# first column is the one explained by the others.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Crude oil ($/barrel)", "Gasoline ($/gallon)", "Jet fuel ($/gallon)", "Tech stock ($)"]
sizes = pd.DataFrame({name: [len(data[name])] for name in columns}, index=["n"])
print(sizes)
y = data[columns[0]].to_numpy(float)
X = data[columns[1:]].to_numpy(float)

## Cointegrating regression

In [ ]:
# The first series regressed on the others, with an intercept: the standard test, which
# also finds cointegration between series whose levels differ by a constant.
X = np.column_stack([np.ones(len(y)), X])
with_intercept = True
fit = sm.OLS(y, X).fit()
residual = fit.resid

## Phillips-Ouliaris test

In [ ]:
# Each series may wander on its own; cointegration means the gap between them, the
# residual above, does not wander but keeps coming back. rho is the slope of
# residual(t) on residual(t-1) through the origin: near 1 the gap persists, well below
# 1 it closes quickly. Phillips and Ouliaris's statistic is Z = n (rho - 1), with n the
# number of pairs, minus a correction for serial correlation in u, the errors of that
# regression; the more negative, the faster the gap closes. H0: no cointegration (the
# gap wanders like a random walk). Its P value comes from a table simulated for this
# statistic (see cointegration_test.py); the usual t or DF tables do not apply,
# because the regression has already made the gap as small as it can.
# The test also assumes each series has a unit root (check with the unit root test
# first), and it only looks for a tie that involves the first column, so the column
# order can change the answer.
lagged, current = residual[:-1], residual[1:]
rho = (lagged @ current) / (lagged @ lagged)
n = len(current)
# The correction: 0.5 n^2 (long-run variance of u - its variance) / sum of
# residual(t-1)^2. The long-run variance adds u's autocovariances up to lag l with
# Bartlett's weights 1 - j / (l + 1); l = n // 100, the bandwidth R's
# tseries::po.test uses by default (lshort = TRUE), so below 100 pairs l = 0 and the
# correction is 0. No Python library has this test, so it is written out.
u = current - rho * lagged
bandwidth = n // 100
extra = sum(2 * (1 - j / (bandwidth + 1)) * (u[j:] @ u[:-j]) / n for j in range(1, bandwidth + 1))
statistic = n * (rho - 1) - 0.5 * n**2 * extra / (lagged @ lagged)
p, note = phillips_ouliaris_p(statistic, regressors=len(columns) - 1, intercept=with_intercept)
test = pd.DataFrame({"rho": [rho], "Bandwidth l": [bandwidth], "Test statistic": [statistic],
                     "P": [p], "": [note]})
print()
print(test.to_string(index=False))

## What the result says

Each of these four price series wanders with no fixed level. Cointegration means that some of them wander together: a combination of them stays stable, so the gap between them keeps closing. The test first explains crude oil by the other three prices, with a constant, and then checks whether what is left over keeps returning to zero.

Over the 50 weeks the test statistic is -45.95, and its P value is below 0.01. The P value is the chance of a statistic this far below zero if the prices had no long-run tie. That chance is small, so the test finds a tie.

Crude oil is tied in the long run to at least one of the other three prices. This run does not say which ones carry the tie. To find out, test crude oil with each price separately.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.